# 🛡️ GRACE: Automated Software Vulnerability Detection via In-Context Learning & Graph Similarity

**Sổ Tay Tái Tạo Kỹ Thuật Cho Môi Trường GPU Kaggle (v1.0.0)**

Sổ tay này triển khai tự động quy trình thử nghiệm phát hiện lỗ hổng phần mềm trên 2 tập dữ liệu benchmark thực tế: **Devign** (`DetectVul/devign`) và **Reveal** (`SensorLLM/Reveal`) bằng việc kết hợp:
1. **Stage 1**: Tự động chuẩn hóa dữ liệu & bóc tách mẫu cân bằng nhãn (**Stratified Slicing**).
2. **Stage 2**: Trích xuất đặc trưng với **CodeT5**, tính ma trận khoảng cách $L_2$, & lựa chọn mẫu ví dụ tối ưu theo công thức **Hybrid Reranking** ($0.7 \times \text{Jaccard} + 0.3 \times \text{GraphSim}$).
3. **Stage 3**: Lắp ráp câu lệnh dẫn hướng theo **Figure 6**, bóc tách nhãn với Parser Regex 4 tầng & bảo vệ tiến trình tuyệt đối bằng **JSONL Realtime Checkpointing**.
4. **Stage 4 & 5**: Đo lường trọn vẹn chỉ số **F1-Score, Precision, Recall** và xuất báo cáo nghiệm thu tự động.

---

## Bước 0: Thiết Lập Thư Mục Làm Việc Từ Dataset Đầu Vào (Dataset Setup)
Sao chép bộ mã nguồn từ thư mục Read-Only (`/kaggle/input/datasets/huuhieu3333/grace-source-code` hoặc `/kaggle/input/grace-source-code`) sang thư mục có quyền Ghi (`/kaggle/working/GRACE`) và chuyển thư mục làm việc.

In [1]:
import os, shutil

# 1. Tự động quét và tìm chính xác thư mục chứa run_pipeline.py trong /kaggle/input
found_src_dir = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'run_pipeline.py' in files:
        found_src_dir = root
        break

dest_dir = '/kaggle/working/GRACE'
if os.path.exists(dest_dir):
    shutil.rmtree(dest_dir)

if found_src_dir:
    print(f'[*] Tìm thấy mã nguồn tại: {found_src_dir}')
    shutil.copytree(found_src_dir, dest_dir)
    print(f'✓ [OK] Đã sao chép toàn bộ mã nguồn sang: {dest_dir}')
else:
    print('[!] ERROR: Không tìm thấy file run_pipeline.py trong /kaggle/input!')

# 2. Chuyển thư mục làm việc về /kaggle/working/GRACE
%cd /kaggle/working/GRACE
!ls -la


[*] Tìm thấy mã nguồn tại: /kaggle/input/datasets/huuhieu3333/grace-source-code/GRACE
✓ [OK] Đã sao chép toàn bộ mã nguồn sang: /kaggle/working/GRACE
/kaggle/working/GRACE
total 80
drwxr-xr-x 3 root root  4096 Aug 21 02:05 .
drwxr-xr-x 3 root root  4096 Aug 21 02:23 ..
-rw-r--r-- 1 root root  3717 Aug 21 02:05 config.py
drwxr-xr-x 3 root root  4096 Aug 21 02:05 data
-rw-r--r-- 1 root root  6983 Aug 21 02:05 data_loader.py
-rw-r--r-- 1 root root 12245 Aug 21 02:05 evaluator.py
-rw-r--r-- 1 root root  8124 Aug 21 02:05 GRACE_Kaggle_Reproduce.ipynb
-rw-r--r-- 1 root root  3525 Aug 21 02:05 metrics.py
-rw-r--r-- 1 root root  5282 Aug 21 02:05 prompt_engine.py
-rw-r--r-- 1 root root   363 Aug 21 02:05 requirements.txt
-rw-r--r-- 1 root root  9985 Aug 21 02:05 retrieval_engine.py
-rw-r--r-- 1 root root  8022 Aug 21 02:05 run_pipeline.py


## Bước 1: Chuẩn Bị Môi Trường & Thư Viện (Environment Setup)
Cài đặt các gói phụ thuộc cơ bản theo chuẩn `requirements.txt`.

In [2]:
!pip install -q -r requirements.txt
print('✓ [Setup Complete] Các thư viện phụ thuộc đã sẵn sàng trên Kaggle!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 37.7 MB/s eta 0:00:00
✓ [Setup Complete] Các thư viện phụ thuộc đã sẵn sàng trên Kaggle!


### Xác Minh Kết Nối Kaggle Secrets
Kiểm tra xem notebook đã được cấp quyền đọc `FPT_API_KEY` và `FPT_BASE_URL` từ Kaggle Secrets chưa.

In [3]:
from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    api_key = user_secrets.get_secret('FPT_API_KEY')
    base_url = user_secrets.get_secret('FPT_BASE_URL')
    print(f'✓ [OK] Đã nạp thành công FPT_API_KEY ({api_key[:6]}...) và FPT_BASE_URL ({base_url})!')
except Exception as e:
    print(f'[!] Cảnh báo nạp Secret: {e}. Vui lòng kiểm tra Add-ons -> Secrets trên Kaggle.')

✓ [OK] Đã nạp thành công FPT_API_KEY (sk-Kbz...) và FPT_BASE_URL (https://mkp-api.fptcloud.com)!


## Bước 2: Chạy Kiểm Thử Khảo Sát Tinh gọn (Warm-up Dry-Run)
Chạy thử nghiệm E2E chế độ Mock trên 10% dữ liệu để xác minh toàn bộ luồng đường ống hoạt động mượt mà.

In [4]:
!python run_pipeline.py --use_mock --sample_ratio 0.1 --experiment_name kaggle_warmup_dryrun

2026-08-21 02:24:06,319 - INFO - Chế độ Mock được kích hoạt: Khởi tạo dữ liệu giả lập chất lượng cao...
2026-08-21 02:24:06,320 - INFO - Hoàn tất chuẩn bị dữ liệu -> Train index: 50 mẫu | Test eval: 20 mẫu.
2026-08-21 02:24:26,654 - INFO - NumExpr defaulting to 4 threads.
2026-08-21 02:24:27,622 - INFO - Đang nạp mô hình Salesforce/codet5-base trên thiết bị cuda...
2026-08-21 02:24:27,883 - INFO - HTTP Request: HEAD https://huggingface.co/Salesforce/codet5-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-21 02:24:27,883 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-08-21 02:24:27,891 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Salesforce/codet5-base/02cd2d31bb7c6d0e4d91156167b2de044989c733/tokenizer_config.json "HTTP/1.1 200 OK"
2026-08-21 02:24:27,899 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-c

## Bước 3: Khởi Chạy Thực Nghiệm Trí Tuệ Nhân Tạo Trên Bộ Dữ Liệu DEVIGN
Kết nối tới **FPT AI Factory API** (DeepSeek-V4-Flash) để đánh giá lỗ hổng bảo mật trên 5% dataset thực tế **DetectVul/devign**.

*(Lưu ý: Bạn cần khai báo biến `FPT_API_KEY` và `FPT_BASE_URL` trong Kaggle Secrets (Add-ons -> Secrets) trước khi chạy. Nhờ cơ chế Checkpointing JSONL, nếu phiên làm việc bị timeout, bạn chỉ cần bấm chạy lại cell này, hệ thống sẽ tiếp tục tại đúng mẫu trước đó!)*

In [5]:
!python run_pipeline.py --dataset DetectVul/devign --sample_ratio 0.05 --experiment_name devign_openweights_5pct

2026-08-21 02:24:34,332 - INFO - Đang kết nối tải dataset 'DetectVul/devign' từ Hugging Face Hub...
2026-08-21 02:24:34,332 - INFO - Phát hiện file dữ liệu đã trích xuất đồ thị CPG cục bộ: data/processed/devign_train_processed.json. Đang nạp...
2026-08-21 02:25:16,874 - INFO - Nạp thành công 21854 mẫu đã có đầy đủ đồ thị Joern từ devign_train_processed.json.
2026-08-21 02:25:17,526 - INFO - Trích xuất 5.0% dataset: Tổng 1091 mẫu (500 Vulnerable, 591 Safe) từ gốc 21854 mẫu.
2026-08-21 02:25:20,046 - INFO - Phát hiện file dữ liệu đã trích xuất đồ thị CPG cục bộ: data/processed/devign_test_processed.json. Đang nạp...
2026-08-21 02:25:24,441 - INFO - Nạp thành công 2732 mẫu đã có đầy đủ đồ thị Joern từ devign_test_processed.json.
2026-08-21 02:25:24,447 - INFO - Trích xuất 5.0% dataset: Tổng 135 mẫu (62 Vulnerable, 73 Safe) từ gốc 2732 mẫu.
2026-08-21 02:25:24,708 - INFO - Hoàn tất chuẩn bị dữ liệu -> Train index: 1091 mẫu | Test eval: 135 mẫu.
2026-08-21 02:25:31,584 - INFO - NumExpr defa

## Bước 4: Khởi Chạy Thực Nghiệm Mở Rộng Trên Bộ Dữ Liệu REVEAL
Thử nghiệm trên bộ dataset Reveal nhằm kiểm chứng khả năng tổng quát hóa (Generalizability) của phương pháp GRACE.

## Bước 5: Phỏng Đoán Báo Cáo Kết Quả Tự Động Xuất Xưởng (Artifact Inspection)
Trình diễn các file kết quả JSON và CSV đã được ghi xuống thư mục `output/`.

In [6]:
import os
import pandas as pd
from pathlib import Path

output_dir = Path('/kaggle/working/output') if os.path.exists('/kaggle/working') else Path('./output')
print('Danh sách tệp nghiệm thu thu được từ hệ thống:')
if output_dir.exists():
    for f in output_dir.glob('*.*'):
        print('  ->', f.name)
else:
    print('Thư mục output chưa được tạo.')

Danh sách tệp nghiệm thu thu được từ hệ thống:
  -> results_kaggle_warmup_dryrun.json
  -> results_devign_openweights_5pct.json
  -> summary_kaggle_warmup_dryrun.csv
  -> summary_devign_openweights_5pct.csv
